# Estimating Forecast Uncertainty in Financial Time Series
## Exploration & Results Notebook

**Chapter mapping:**
| Notebook section | Thesis section |
|---|---|
| 1. Imports & Config | §3 Methodology |
| 2. EDA | §3 — Data Description |
| 3. Preprocessing | §3 — Feature Engineering |
| 4. Model Training | §3 — LSTM / GRU Ensemble |
| 5. Conformal Inference | §3 — EnbPI & AgACI |
| 6. Point Forecast Accuracy | §4.2 Results |
| 7. Prediction Interval Quality | §4.3 Results |
| 8. Adaptive Behaviour | §4.4 Results |

## 1 — Imports & Config  (§3 Methodology)

In [1]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append('..')

from src.processor import DataProcessor
from src.models import EnsembleWrapper
from src.conformal import get_residuals, apply_enbpi
from src.evaluation import (
    rmse, mae,
    coverage, interval_width,
    winkler_score, conditional_coverage
)
from src.visualization import plot_rolling_coverage

with open('../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

ALPHA = config['conformal']['alpha']

plt.style.use('ggplot')
print(f"Loaded config: alpha={ALPHA}")

I0000 00:00:1777368571.284192       8 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777368571.287902       8 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777368572.001172       8 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1777368574.523245       8 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777368574.526026       8 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Loaded config: alpha=0.1


## 2 — Exploratory Data Analysis  (§3 — Data Description)

In [2]:
df = pd.read_csv(f"../{config['data']['file_path']}")
col = config['data']['target_column']

df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=[col])
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').set_index('Date')

df['Returns'] = df[col].pct_change()
df = df.dropna()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))

ax[0].plot(df[col])
ax[0].set_title("Price Series")

sns.histplot(x=df['Returns'], bins=100, kde=True, ax=ax[1])
ax[1].set_title("Returns Distribution (Fat Tails)")

plt.show()

df.describe().T

,count,mean,std,min,25%,50%,75%,max
Close,2515.0,3006.349900,901.213023,1741.890015,2125.030029,2798.290039,3907.449951,4796.560059
Returns,2515.0,0.000443,0.011174,-0.119841,-0.003788,0.000598,0.005661,0.093828


## 3 — Data Preprocessing  (§3 — Feature Engineering)

Sliding-window sequences of length `window_size` are created from the Close price.
The scaler is **fit on the training set only** to prevent data leakage into calibration/test.

In [3]:
dp = DataProcessor(window=config['data']['window_size'])

series = dp.load_series(f"../{config['data']['file_path']}", column=col)
X, y = dp.create_sequences(series)

(X_train, y_train), (X_cal, y_cal), (X_test, y_test) = dp.split_data(
    X, y,
    train_p=config['data']['train_split'],
    cal_p=config['data']['cal_split']
)

# Fit scaler ONLY on training
dp.fit_scaler(np.vstack((X_train.reshape(-1,1), y_train.reshape(-1,1))))

X_train = dp.transform(X_train)
y_train = dp.transform(y_train)
X_cal   = dp.transform(X_cal)
y_cal   = dp.transform(y_cal)
X_test  = dp.transform(X_test)
y_test  = dp.transform(y_test)

# LSTM/GRU expect 3D inputs: (samples, timesteps, features)
X_train = X_train[..., np.newaxis]
X_cal   = X_cal[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

print(X_train.shape, X_cal.shape, X_test.shape)

(1491, 30, 1) (497, 30, 1) (498, 30, 1)


## 4 — Model Training  (§3 — LSTM / GRU Bootstrap Ensemble)

Each ensemble member is trained on a bootstrap resample of the training set.
The mean prediction across B members becomes the point forecast.

In [4]:
print("Training LSTM...")
lstm = EnsembleWrapper(
    "LSTM",
    B=config['model']['ensemble_size'],
    hidden_units=tuple(config['model']['hidden_units']),
    dropout_rate=config['model']['dropout_rate'],
)
lstm.fit(X_train, y_train, epochs=config['model']['epochs'])

print("Training GRU...")
gru = EnsembleWrapper(
    "GRU",
    B=config['model']['ensemble_size'],
    hidden_units=tuple(config['model']['hidden_units']),
    dropout_rate=config['model']['dropout_rate'],
)
gru.fit(X_train, y_train, epochs=config['model']['epochs'])

Training LSTM...
Training LSTM member 1/5


E0000 00:00:1777368575.430528       8 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training LSTM member 2/5


Training LSTM member 3/5


Training LSTM member 4/5


...

Training GRU...
Training GRU member 1/5


Training GRU member 2/5


Training GRU member 3/5


Training GRU member 4/5


Training GRU member 5/5


## 5 — Conformal Inference  (§3 — EnbPI & AgACI)

**EnbPI** uses a fixed quantile from calibration-set residuals.
**AgACI** updates the quantile at every test step using a rolling window of recent residuals with exponential decay weighting.

In [5]:
# calibration predictions
lstm_cal_pred = lstm.predict(X_cal)
gru_cal_pred  = gru.predict(X_cal)

lstm_res = get_residuals(y_cal, lstm_cal_pred)
gru_res  = get_residuals(y_cal, gru_cal_pred)

# test predictions
lstm_pred = lstm.predict(X_test)
gru_pred  = gru.predict(X_test)

# EnbPI
l_enb, u_enb = apply_enbpi(lstm_pred, lstm_res, alpha=ALPHA)
lg_enb, ug_enb = apply_enbpi(gru_pred, gru_res, alpha=ALPHA)

In [6]:
def agaci_sequential(preds, y_true, init_residuals, alpha, window=50):
    history = list(init_residuals)

    lower, upper = [], []

    for i in range(len(preds)):
        recent = np.array(history[-window:])

        if len(recent) == 0:
            q = 0
        else:
            q = np.quantile(recent, 1 - alpha)

        lower.append(preds[i] - q)
        upper.append(preds[i] + q)

        # update AFTER prediction (causal)
        history.append(abs(y_true[i] - preds[i]))

    return np.array(lower), np.array(upper)


l_ag, u_ag = agaci_sequential(lstm_pred, y_test, lstm_res, ALPHA)
lg_ag, ug_ag = agaci_sequential(gru_pred, y_test, gru_res, ALPHA)

## 6 — §4.2  Point Forecast Accuracy

Metrics: **RMSE**, **MAE**, **MAPE** compared against a persistence baseline.
The **Diebold-Mariano test** checks whether the difference between LSTM and GRU is statistically significant.

In [7]:
baseline = X_test[:, -1, 0]

table = pd.DataFrame([
    {
        "Model": "LSTM",
        "RMSE": rmse(y_test, lstm_pred),
        "MAE": mae(y_test, lstm_pred)
    },
    {
        "Model": "GRU",
        "RMSE": rmse(y_test, gru_pred),
        "MAE": mae(y_test, gru_pred)
    },
    {
        "Model": "Baseline",
        "RMSE": rmse(y_test, baseline),
        "MAE": mae(y_test, baseline)
    }
])

table

,Model,RMSE,MAE
0,LSTM,1.142203,1.012461
1,GRU,0.884019,0.808493
2,Baseline,0.132831,0.100861


## 7 — §4.3  Prediction Interval Quality

| Metric | What it captures |
|---|---|
| **Coverage** | Fraction of actuals inside the 90% interval |
| **Coverage Deviation** | \|empirical − 90%\| — smaller is better |
| **Avg Width** | Average interval size — smaller = more efficient |
| **Winkler Score** | Proper scoring rule: penalises width *and* violations |
| **Kupiec p-value** | χ² test: p > 0.05 → interval is well-calibrated |

In [8]:
rows = []

for name, l, u in [
    ("LSTM EnbPI", l_enb, u_enb),
    ("GRU EnbPI", lg_enb, ug_enb),
    ("LSTM AgACI", l_ag, u_ag),
    ("GRU AgACI", lg_ag, ug_ag),
]:
    rows.append({
        "Model": name,
        "Coverage": coverage(y_test, l, u),
        "Width": interval_width(l, u),
        "Winkler": winkler_score(y_test, l, u, ALPHA)
    })

pd.DataFrame(rows)

,Model,Coverage,Width,Winkler
0,LSTM EnbPI,0.903614,3.482957,3.860401
1,GRU EnbPI,0.897590,2.579494,2.857787
2,LSTM AgACI,0.771084,2.738067,3.265883
3,GRU AgACI,0.799197,2.133626,2.477290


## 8 — §4.4  Adaptive Behaviour — Rolling & Conditional Coverage

**Rolling coverage** shows whether AgACI tracks the 90% target locally over time,
whereas EnbPI can drift during volatile regimes.

**Conditional coverage** splits the test set into high-volatility and low-volatility
periods to formally compare regime-specific calibration.

In [9]:
returns = np.diff(y_test.flatten())

cc = conditional_coverage(y_test.flatten(), l_enb, u_enb, returns)

cc

{'high_vol_coverage': 0.96,
 'low_vol_coverage': 0.8847184986595175,
 'high_vol_n': 125,
 'low_vol_n': 373}

In [10]:
plot_rolling_coverage(
    y_test.flatten(),
    l_enb, u_enb,
    l_ag, u_ag,
    model_name="LSTM"
)